This notebook builds an AI-based Resume Analyzer that compares a resume with a job description to calculate an ATS score. It uses NLP techniques like TF-IDF vectorization and cosine similarity to measure relevance, along with keyword/skill extraction to identify matched and missing skills.

The output includes an ATS score, skill comparison, and suggestions to improve the resume for better job alignment.

Importing required libraries
Resume text extraction
Job description input
Text preprocessing
TF-IDF vectorization
Cosine similarity (ATS score calculation)
Skill extraction (e.g., using keyword techniques)
Skill comparison (matched vs missing)
Weighted scoring system
Final output and suggestions

IMPORTING LIBRARIES

In [45]:
!pip install PyPDF2


In [46]:
import pandas as pd
import numpy as np

## for pdf processing (extrating text frm pdf)
import PyPDF2
import re  ## for text cleaning

EXTRACT RESUME TEXT

In [47]:
def extract_resume_text(file_path):
    text = ""
    with open(file_path,'rb') as file:
        reader = PyPDF2.PdfReader(file)
        
        for page in reader.pages:
            text += page.extract_text()

    return text

In [48]:
resume_text = extract_resume_text("resume.pdf")
print(resume_text[:500])

Anurag Tiwari
/githubAnuragtiwari05| /linkedinanuragtiwari05|
/envel⌢peaanuragtiwari1625@gmail.com|
About me
A dedicated Web Developer with practical experience in developing full-stack applications using Next.js,
backend APIs, and modern web technologies. Skilled in building secure authentication systems, integrating
third-party services, and creating user-friendly interfaces. Eager to grow professionally and contribute
effectively to organizational goals.
Education
MU - Slrtce :- persuing comp


INPUT JOB DESCRIPTION

In [49]:
job_description = """
We are looking for a Python Developer with experience in machine learning,
data analysis, and web development. The candidate should have knowledge of
Python, Pandas, NumPy, Scikit-learn, and REST APIs. Experience with
FastAPI or Flask is a plus.
"""

In [50]:
print(job_description)



We are looking for a Python Developer with experience in machine learning,
data analysis, and web development. The candidate should have knowledge of
Python, Pandas, NumPy, Scikit-learn, and REST APIs. Experience with
FastAPI or Flask is a plus.



Text Cleaning & Preprocessing

In [51]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [52]:
clean_res = clean_text(resume_text)
clean_jd = clean_text(job_description)

In [53]:
print(clean_res[:500])
print("\n--- JOB DESCRIPTION ---\n")
print(clean_jd)


anurag tiwari githubanuragtiwari linkedinanuragtiwari envelpeaanuragtiwarigmailcom about me a dedicated web developer with practical experience in developing fullstack applications using nextjs backend apis and modern web technologies skilled in building secure authentication systems integrating thirdparty services and creating userfriendly interfaces eager to grow professionally and contribute effectively to organizational goals education mu slrtce persuing computer science in be skills languag

--- JOB DESCRIPTION ---

we are looking for a python developer with experience in machine learning data analysis and web development the candidate should have knowledge of python pandas numpy scikitlearn and rest apis experience with fastapi or flask is a plus


TF-IDF Vectorization

In [54]:
from sklearn.feature_extraction.text import TfidfVectorizer
vector = TfidfVectorizer(stop_words='english')

In [55]:
texts = [clean_res,clean_jd]
tfidf_matrix = vector.fit_transform(texts)

In [56]:
print(tfidf_matrix)

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 175 stored elements and shape (2, 166)>
  Coords	Values
  (0, 7)	0.055156430234577676
  (0, 153)	0.055156430234577676
  (0, 68)	0.055156430234577676
  (0, 85)	0.055156430234577676
  (0, 57)	0.055156430234577676
  (0, 47)	0.055156430234577676
  (0, 163)	0.19622127193474534
  (0, 51)	0.03924425438694907
  (0, 115)	0.055156430234577676
  (0, 58)	0.03924425438694907
  (0, 52)	0.055156430234577676
  (0, 65)	0.055156430234577676
  (0, 12)	0.11031286046915535
  (0, 159)	0.11031286046915535
  (0, 98)	0.2206257209383107
  (0, 19)	0.165469290703733
  (0, 9)	0.03924425438694907
  (0, 92)	0.055156430234577676
  (0, 149)	0.055156430234577676
  (0, 138)	0.055156430234577676
  (0, 22)	0.11031286046915535
  (0, 134)	0.11031286046915535
  (0, 17)	0.165469290703733
  (0, 147)	0.055156430234577676
  (0, 76)	0.055156430234577676
  :	:
  (0, 105)	0.055156430234577676
  (0, 95)	0.055156430234577676
  (0, 72)	0.055156430234577676
  (0, 34)	0.05515

In [57]:
tfidf_matrix.shape   ##(2=document , 187 = unique words identifies)

(2, 166)

Cosine Similarity (ATS Score)  - FEATURE 1 

In [58]:
from sklearn.metrics.pairwise import cosine_similarity
similarity_score = cosine_similarity(tfidf_matrix[0:1],tfidf_matrix[1:2])

In [59]:
ats_score = similarity_score[0][0]*100
print(f"ATS Score: {ats_score:.2f}%")

ATS Score: 9.71%


Skill Extraction (NLP) - FEATURE 2

In [60]:
import json

with open("skills.json", "r") as f:
    skill_data = json.load(f)

all_skills = []
for category in skill_data.values():
    all_skills.extend(category)


def extract_skills(text, skill_list):
    found_skills = []
    
    for skill in skill_list:
        if skill in text:
            found_skills.append(skill)
    
    return list(set(found_skills))
resume_skills = extract_skills(clean_res, all_skills)
jd_skills = extract_skills(clean_jd, all_skills)

In [61]:
print("Resume Skills:", resume_skills)
print("------------------")
print("Job Description Skills:", jd_skills)


Resume Skills: ['nextjs', 'postgresql', 'fastapi', 'github', 'tailwind', 'mongodb', 'python', 'nodejs', 'mysql', 'html', 'css', 'javascript', 'express', 'react', 'git', 'java']
------------------
Job Description Skills: ['fastapi', 'numpy', 'python', 'pandas', 'flask', 'machine learning']


Skill Comparison (Matched vs Missing) - FEATURE 3

In [62]:
resume_set = set(resume_skills)
jd_set = set(jd_skills)

matched_skills = list(resume_set.intersection(jd_set))
missing_skills = list(jd_set-resume_set)

In [63]:
print(matched_skills)
print("------------------------")
print(missing_skills)

['python', 'fastapi']
------------------------
['flask', 'machine learning', 'pandas', 'numpy']


Weighted Score Calculation - FEATURE 4

In [64]:
skill_match_score = (len(matched_skills) / len(jd_skills)) * 100
final_score = (0.6 * ats_score) + (0.4 * skill_match_score)


In [65]:
print(f"ATS Score: {ats_score:.2f}%")
print(f"Skill Match Score: {skill_match_score:.2f}%")
print(f"Final Score: {final_score:.2f}%")


ATS Score: 9.71%
Skill Match Score: 33.33%
Final Score: 19.16%
